# Notebook 01: Introduction to LangGraph - Building Your First Graph

Welcome to the LangGraph tutorial series! In this notebook, you'll learn the fundamentals of LangGraph and build your first stateful AI workflow.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand what LangGraph is and when to use it
2. Create a basic StateGraph with TypedDict
3. Define node functions that process state
4. Connect nodes with edges to create workflows
5. Compile and execute graphs
6. Debug and understand graph execution flow

## Prerequisites

- Python 3.12 or 3.13 installed
- Basic Python knowledge (functions, dictionaries, type hints)
- Jupyter Notebook environment

## Estimated Time

90-120 minutes

## Complexity Level

Beginner (1/5)

## 1. Introduction: What is LangGraph?

### The Challenge

Modern AI applications often need to:
- Maintain state across multiple steps
- Make decisions based on previous outputs
- Coordinate multiple AI agents or tools
- Handle complex, branching workflows

Traditional sequential code or simple chains can become messy and hard to maintain for these scenarios.

### The Solution: LangGraph

**LangGraph** is a framework for building **stateful, multi-actor applications** with LLMs. It provides:

- **Graph-based architecture**: Define workflows as nodes (steps) and edges (transitions)
- **State management**: Automatic state passing between nodes
- **Flexibility**: Support for conditional routing, loops, and parallel execution
- **Persistence**: Save and resume workflows
- **Human-in-the-loop**: Pause for human input when needed

### When to Use LangGraph

Use LangGraph when you need:

✅ **Multi-step workflows** with state persistence  
✅ **Conditional branching** based on intermediate results  
✅ **Cyclic flows** where steps may repeat  
✅ **Agent coordination** with multiple AI actors  
✅ **Human oversight** in critical workflows  

Don't use LangGraph for:

❌ Simple, one-shot LLM calls (use LangChain or direct API)  
❌ Purely sequential, linear workflows (use chains)  
❌ When state management isn't needed  

### Core Concepts

1. **State**: A shared data structure (like a dictionary) passed between nodes
2. **Nodes**: Functions that receive state, process it, and return updates
3. **Edges**: Connections that define how execution flows between nodes
4. **Graph**: The complete workflow definition
5. **Compilation**: Converting the graph definition into an executable workflow

> **Note**: `MessageGraph` has been removed as of LangGraph 1.1.x. Use `StateGraph` with `MessagesState` instead.

## 2. Setup

Let's import the necessary libraries and set up our environment.

### Installation

```bash
# Using uv (recommended)
# uv venv --python 3.12 && source .venv/bin/activate && uv pip install -e .

# Or traditional pip:
# pip install langgraph>=1.1.9 langchain-openai>=1.2.1
```

In [ ]:
# Core imports
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# For pretty printing
import json

print("✅ Imports successful!")
print("📦 Ready to build your first graph!")

## 3. Your First Graph: Hello World

Let's build the simplest possible graph with two nodes.

### Step 1: Define the State Schema

The state is a shared data structure that flows through your graph. We define it using `TypedDict`.

In [ ]:
class State(TypedDict):
    """The state that will flow through our graph."""
    text: str  # A simple text field

**What's happening here?**

- We create a `State` class that inherits from `TypedDict`
- This defines the structure of data that will be passed between nodes
- In this case, our state contains a single field: `text` of type `str`
- This provides type safety and auto-completion in your IDE

## 💡 LangGraph 1.1.9: MessagesState Built-In

LangGraph now ships a built-in `MessagesState` TypedDict that includes `messages: Annotated[list, add_messages]` — no need to define it manually when working with message-based agents:

```python
from langgraph.graph import MessagesState  # ← new in 1.1.x

# Equivalent to:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
```

We cover MessagesState in depth in **Notebook 03** when building tool-using agents.

### Step 2: Define Node Functions

Nodes are functions that receive the current state and return updates to it.

In [ ]:
def node_a(state: State) -> dict:
    """First node: appends 'a' to the text."""
    print("🔵 Node A executing...")
    return {"text": state["text"] + "a"}

def node_b(state: State) -> dict:
    """Second node: appends 'b' to the text."""
    print("🟢 Node B executing...")
    return {"text": state["text"] + "b"}

**Key Points:**

- Node functions receive the **full state** as input
- They return a **dictionary with updates** to apply to the state
- The returned dictionary doesn't need to include all state keys—only what changed
- LangGraph automatically merges updates into the current state

### Step 3: Build the Graph

Now we'll create a `StateGraph` and add our nodes and edges.

In [ ]:
# Create a new graph with our State schema
graph = StateGraph(State)

# Add nodes to the graph
graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)

# Define the flow: START -> node_a -> node_b -> END
graph.add_edge(START, "node_a")
graph.add_edge("node_a", "node_b")
graph.add_edge("node_b", END)

print("✅ Graph structure defined!")

**Understanding the code:**

1. `StateGraph(State)`: Creates a new graph with our state schema
2. `add_node(name, function)`: Registers a node with a unique name
3. `add_edge(from, to)`: Creates a directed edge from one node to another
4. `START`: Special constant marking the entry point
5. `END`: Special constant marking the exit point

**Execution flow:**
```
START → node_a → node_b → END
```

### Step 4: Compile and Execute

Before we can run the graph, we need to compile it.

In [ ]:
# Compile the graph into an executable
app = graph.compile()

# Run the graph with an initial state
initial_state = {"text": ""}
result = app.invoke(initial_state)

print("\n" + "="*50)
print("📊 Final Result:")
print(json.dumps(result, indent=2))
print("="*50)

**Expected Output:**
```
🔵 Node A executing...
🟢 Node B executing...

==================================================
📊 Final Result:
{
  "text": "ab"
}
==================================================
```

**What happened?**

1. Started with `{"text": ""}`
2. `node_a` appended "a" → `{"text": "a"}`
3. `node_b` appended "b" → `{"text": "ab"}`
4. Final result: `{"text": "ab"}`

Congratulations! You've just built and executed your first LangGraph workflow! 🎉

## 4. Example 2: Text Processing Pipeline

Let's build something more practical—a text processing pipeline that:
1. Converts text to lowercase
2. Removes punctuation
3. Counts words

This demonstrates a realistic multi-step workflow.

In [ ]:
import re

# Define a more complex state
class TextState(TypedDict):
    original: str      # Original text
    processed: str     # Processed text
    word_count: int    # Word count result

# Node 1: Convert to lowercase
def to_lowercase(state: TextState) -> dict:
    """Convert text to lowercase."""
    print("🔄 Converting to lowercase...")
    lowercased = state["original"].lower()
    return {"processed": lowercased}

# Node 2: Remove punctuation
def remove_punctuation(state: TextState) -> dict:
    """Remove all punctuation from text."""
    print("🔄 Removing punctuation...")
    cleaned = re.sub(r'[^\w\s]', '', state["processed"])
    return {"processed": cleaned}

# Node 3: Count words
def count_words(state: TextState) -> dict:
    """Count the number of words."""
    print("🔄 Counting words...")
    words = state["processed"].split()
    count = len(words)
    return {"word_count": count}

# Build the graph
text_graph = StateGraph(TextState)

text_graph.add_node("lowercase", to_lowercase)
text_graph.add_node("remove_punct", remove_punctuation)
text_graph.add_node("count", count_words)

text_graph.add_edge(START, "lowercase")
text_graph.add_edge("lowercase", "remove_punct")
text_graph.add_edge("remove_punct", "count")
text_graph.add_edge("count", END)

# Compile
text_app = text_graph.compile()

print("✅ Text processing pipeline ready!")

In [ ]:
# Test the pipeline
test_text = "Hello, World! This is LangGraph. It's amazing!!"

result = text_app.invoke({
    "original": test_text,
    "processed": "",
    "word_count": 0
})

print("\n" + "="*60)
print("📝 Text Processing Results:")
print("="*60)
print(f"Original:  {result['original']}")
print(f"Processed: {result['processed']}")
print(f"Words:     {result['word_count']}")
print("="*60)

## 5. Example 3: Simple Calculator Chain

Let's build a calculator that performs a series of operations, demonstrating state accumulation.

In [ ]:
class CalcState(TypedDict):
    value: float      # Current value
    operations: list  # Log of operations performed

def add_10(state: CalcState) -> dict:
    """Add 10 to the current value."""
    new_value = state["value"] + 10
    operations = state["operations"] + [f"Added 10: {state['value']} + 10 = {new_value}"]
    print(f"➕ Adding 10: {state['value']} → {new_value}")
    return {"value": new_value, "operations": operations}

def multiply_by_2(state: CalcState) -> dict:
    """Multiply the current value by 2."""
    new_value = state["value"] * 2
    operations = state["operations"] + [f"Multiplied by 2: {state['value']} × 2 = {new_value}"]
    print(f"✖️  Multiplying by 2: {state['value']} → {new_value}")
    return {"value": new_value, "operations": operations}

def subtract_5(state: CalcState) -> dict:
    """Subtract 5 from the current value."""
    new_value = state["value"] - 5
    operations = state["operations"] + [f"Subtracted 5: {state['value']} - 5 = {new_value}"]
    print(f"➖ Subtracting 5: {state['value']} → {new_value}")
    return {"value": new_value, "operations": operations}

# Build calculator graph
calc_graph = StateGraph(CalcState)

calc_graph.add_node("add", add_10)
calc_graph.add_node("multiply", multiply_by_2)
calc_graph.add_node("subtract", subtract_5)

# Chain: START → add → multiply → subtract → END
calc_graph.add_edge(START, "add")
calc_graph.add_edge("add", "multiply")
calc_graph.add_edge("multiply", "subtract")
calc_graph.add_edge("subtract", END)

calc_app = calc_graph.compile()

print("✅ Calculator ready!")

In [ ]:
# Test the calculator: (5 + 10) × 2 - 5 = ?
result = calc_app.invoke({
    "value": 5.0,
    "operations": []
})

print("\n" + "="*60)
print("🧮 Calculator Results:")
print("="*60)
print(f"Starting value: 5.0")
print(f"Final value: {result['value']}")
print("\nOperation Log:")
for op in result['operations']:
    print(f"  • {op}")
print("="*60)

## 6. Hands-on Exercise: Recipe Formatter

Now it's your turn! Build a graph that processes a recipe through these steps:

1. **Format Ingredients**: Capitalize each ingredient and add bullet points
2. **Format Steps**: Number each step
3. **Validate**: Check that the recipe has at least 1 ingredient and 1 step
4. **Compile Output**: Create a nicely formatted recipe string

### State Schema

```python
class RecipeState(TypedDict):
    ingredients: list[str]     # List of ingredients
    steps: list[str]           # List of cooking steps
    formatted_ingredients: str # Formatted ingredients output
    formatted_steps: str       # Formatted steps output
    is_valid: bool            # Validation result
    final_recipe: str         # Complete formatted recipe
```

### Your Task

Complete the implementation below:

In [ ]:
class RecipeState(TypedDict):
    ingredients: list
    steps: list
    formatted_ingredients: str
    formatted_steps: str
    is_valid: bool
    final_recipe: str

def format_ingredients(state: RecipeState) -> dict:
    """Format ingredients with bullet points and capitalization."""
    # TODO: Implement this function
    # Hint: Use list comprehension and string formatting
    pass

def format_steps(state: RecipeState) -> dict:
    """Number each step."""
    # TODO: Implement this function
    # Hint: Use enumerate()
    pass

def validate_recipe(state: RecipeState) -> dict:
    """Validate that recipe has ingredients and steps."""
    # TODO: Implement this function
    # Hint: Check list lengths
    pass

def compile_output(state: RecipeState) -> dict:
    """Create final formatted recipe."""
    # TODO: Implement this function
    # Hint: Combine formatted_ingredients and formatted_steps
    pass

# TODO: Build your graph here
# recipe_graph = StateGraph(RecipeState)
# ... add nodes and edges ...
# recipe_app = recipe_graph.compile()

### Test Your Implementation

Run this cell to test your recipe formatter:

In [ ]:
# Test data
test_recipe = {
    "ingredients": ["flour", "eggs", "milk", "butter"],
    "steps": [
        "Mix flour and eggs",
        "Add milk gradually",
        "Melt butter and fold in"
    ],
    "formatted_ingredients": "",
    "formatted_steps": "",
    "is_valid": False,
    "final_recipe": ""
}

# Uncomment when ready:
# result = recipe_app.invoke(test_recipe)
# print(result['final_recipe'])

<details>
<summary><b>Click here for the solution</b></summary>

```python
def format_ingredients(state: RecipeState) -> dict:
    formatted = "\n".join([f"• {ing.capitalize()}" for ing in state["ingredients"]])
    return {"formatted_ingredients": formatted}

def format_steps(state: RecipeState) -> dict:
    formatted = "\n".join([f"{i+1}. {step}" for i, step in enumerate(state["steps"])])
    return {"formatted_steps": formatted}

def validate_recipe(state: RecipeState) -> dict:
    is_valid = len(state["ingredients"]) > 0 and len(state["steps"]) > 0
    return {"is_valid": is_valid}

def compile_output(state: RecipeState) -> dict:
    if state["is_valid"]:
        final = f"INGREDIENTS:\n{state['formatted_ingredients']}\n\nSTEPS:\n{state['formatted_steps']}"
    else:
        final = "Invalid recipe: missing ingredients or steps"
    return {"final_recipe": final}

recipe_graph = StateGraph(RecipeState)
recipe_graph.add_node("format_ing", format_ingredients)
recipe_graph.add_node("format_steps", format_steps)
recipe_graph.add_node("validate", validate_recipe)
recipe_graph.add_node("compile", compile_output)

recipe_graph.add_edge(START, "format_ing")
recipe_graph.add_edge("format_ing", "format_steps")
recipe_graph.add_edge("format_steps", "validate")
recipe_graph.add_edge("validate", "compile")
recipe_graph.add_edge("compile", END)

recipe_app = recipe_graph.compile()
```
</details>

## 7. Key Takeaways

Congratulations on completing Notebook 01! Let's review what you've learned:

### Core Concepts

1. **StateGraph**: The main class for building workflows
2. **State Schema**: Use `TypedDict` to define your state structure
3. **Nodes**: Functions that process and update state
4. **Edges**: Define the flow between nodes
5. **Compilation**: Convert graph definition to executable
6. **Invocation**: Run the graph with initial state

### Best Practices

✅ **Use descriptive node names** - Makes graphs easier to understand  
✅ **Keep nodes focused** - Each node should do one thing well  
✅ **Type your state** - Helps catch errors early  
✅ **Return only changes** - Nodes don't need to return the full state  
✅ **Add logging** - Print statements help debug execution  

### Common Pitfalls

❌ Forgetting to compile before invoking  
❌ Not connecting START or END nodes  
❌ Mutating state directly instead of returning updates  
❌ Missing required state fields in initial invocation  
❌ Creating circular dependencies without conditional logic  

### What's Next?

In **Notebook 02: Dynamic Routing with Conditional Edges**, you'll learn:

- How to make decisions based on state
- Conditional edges for dynamic routing
- Building branching workflows
- Graph visualization and debugging

This will unlock much more powerful workflow patterns!

## 8. Further Reading

To deepen your understanding, explore these resources:

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [StateGraph API Reference](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.StateGraph)
- [LangGraph Quick Start](https://langchain-ai.github.io/langgraph/tutorials/introduction/)
- [TypedDict Documentation](https://docs.python.org/3/library/typing.html#typing.TypedDict)

### Additional Exercises

Want more practice? Try building these:

1. **String Transformer**: A graph that reverses, uppercases, and counts characters in a string
2. **Data Validator**: Process user input through validation, sanitization, and formatting steps
3. **Math Quiz**: Generate random math problems and track score through multiple steps

---

**Ready for more?** Continue to [Notebook 02: Dynamic Routing with Conditional Edges](02_conditional_routing.ipynb) to learn about decision-making in graphs!